# Data Preparation & Feature Engineering

In this notebook, I will clean and prepare the restaurant metadata and review datasets for clustering.

I will handle missing values, duplicates, inconsistencies, and data types while transforming review-level data into restaurant-level features. I will also perform feature engineering to extract meaningful information that can help identify similarities and differences between restaurants.

The objective is to create a clean, structured, and informative dataset that can be used in the subsequent stages of unsupervised machine learning.

## 1. Imports & Project Setup

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Path define
project_root = Path.cwd().parent
data_path = project_root / "1. data"
raw_data_path = data_path / "raw"

In [ ]:
# Verifying the above cells
print("Project Root: ", project_root)
print("Raw Data Path: ", raw_data_path)
print("\nFiles in Raw Data Folder: ")
print(list(raw_data_path.iterdir()))

Project Root:  d:\Documents\Projects\zomato-clustering-model
Raw Data Path:  d:\Documents\Projects\zomato-clustering-model\1. data\raw

Files in Raw Data Folder: 
[WindowsPath('d:/Documents/Projects/zomato-clustering-model/1. data/raw/Zomato Restaurant names and Metadata.csv'), WindowsPath('d:/Documents/Projects/zomato-clustering-model/1. data/raw/Zomato Restaurant reviews.csv')]


## 2. Data Loading

In [4]:
# File Paths
metadata_file = raw_data_path / "Zomato Restaurant names and Metadata.csv" 
reviews_file = raw_data_path / "Zomato Restaurant reviews.csv" 

# Loading data
metadata_df = pd.read_csv(metadata_file)
reviews_df = pd.read_csv(reviews_file)

In [5]:
# Dataset dimension (Verification)
print("Metadata Shape: ", metadata_df.shape)
print("Reviews Shape: ", reviews_df.shape)

Metadata Shape:  (105, 6)
Reviews Shape:  (10000, 7)


In [6]:
print("Metadata Columns: \n", metadata_df.columns.tolist())
print("Review Columns: \n", reviews_df.columns.tolist())

Metadata Columns: 
 ['Name', 'Links', 'Cost', 'Collections', 'Cuisines', 'Timings']
Review Columns: 
 ['Restaurant', 'Reviewer', 'Review', 'Rating', 'Metadata', 'Time', 'Pictures']


In [7]:
# Previewing Data
display(metadata_df.head())
display(reviews_df.head())

,Name,Links,Cost,Collections,Cuisines,Timings
0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
1,Paradise,https://www.zomato.com/hyderabad/paradise-gach...,800,Hyderabad's Hottest,"Biryani, North Indian, Chinese",11 AM to 11 PM
2,Flechazo,https://www.zomato.com/hyderabad/flechazo-gach...,"1,300","Great Buffets, Hyderabad's Hottest","Asian, Mediterranean, North Indian, Desserts","11:30 AM to 4:30 PM, 6:30 PM to 11 PM"
3,Shah Ghouse Hotel & Restaurant,https://www.zomato.com/hyderabad/shah-ghouse-h...,800,Late Night Restaurants,"Biryani, North Indian, Chinese, Seafood, Bever...",12 Noon to 2 AM
4,Over The Moon Brew Company,https://www.zomato.com/hyderabad/over-the-moon...,"1,200","Best Bars & Pubs, Food Hygiene Rated Restauran...","Asian, Continental, North Indian, Chinese, Med...","12noon to 11pm (Mon, Tue, Wed, Thu, Sun), 12no..."


,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0
3,Beyond Flavours,Swapnil Sarkar,Soumen das and Arun was a great guy. Only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0
4,Beyond Flavours,Dileep,Food is good.we ordered Kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0


## 3. Data Cleaning

### 3.1 Metadata Dataset

I will begin by cleaning the restaurant metadata dataset. This involves preparing the restaurant attributes, handling inconsistencies, and ensuring that the data is suitable for feature engineering and subsequent clustering.

In [8]:
metadata_clean = metadata_df.copy()

In [9]:
# Checking for spaces
metadata_clean["Name"].str.strip().ne(metadata_clean["Name"]).sum()

np.int64(0)

In [10]:
metadata_clean["Collections"] = metadata_clean["Collections"].str.strip()           # Cleaning spaces if any

In [11]:
metadata_clean["Cuisines"] = metadata_clean["Cuisines"].str.strip()         # Cleaning spaces if any

#### 3.1.1 Removing Timings

The Timings column contains inconsistent formats, multiple time intervals, and day-specific schedules, making reliable standardization difficult.

Since opening hours are not a primary feature of restaurant similarity for this project, I will exclude this column from the working dataset while preserving it in the original dataset for reference.

In [12]:
metadata_clean = metadata_clean.drop(columns=["Timings"])

#### 3.1.2 Removing Links

The Links column contains URLs pointing to restaurant pages. Since these links do not provide meaningful features for restaurant segmentation, I will remove this column from the working dataset while retaining the original data for reference.

In [13]:
metadata_clean = metadata_clean.drop(columns=["Links"])

In [14]:
display(metadata_clean.head())          # Cleaned Metadata Preview

,Name,Cost,Collections,Cuisines
0,Beyond Flavours,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I..."
1,Paradise,800,Hyderabad's Hottest,"Biryani, North Indian, Chinese"
2,Flechazo,"1,300","Great Buffets, Hyderabad's Hottest","Asian, Mediterranean, North Indian, Desserts"
3,Shah Ghouse Hotel & Restaurant,800,Late Night Restaurants,"Biryani, North Indian, Chinese, Seafood, Bever..."
4,Over The Moon Brew Company,"1,200","Best Bars & Pubs, Food Hygiene Rated Restauran...","Asian, Continental, North Indian, Chinese, Med..."


### 3.2 Reviews Dataset

I will now clean the restaurant reviews dataset. Since each row represents an individual review, I will address duplicate records, placeholder entries, missing values, and data types while preserving information that can contribute to restaurant-level features.

In [15]:
reviews_clean = reviews_df.copy()            # Copying

#### 3.2.1 Removing Duplicates

In [16]:
reviews_clean = reviews_clean.drop_duplicates()

In [17]:
# Removing placeholders
placeholder = (
    reviews_clean["Reviewer"].isna() &
    reviews_clean["Review"].isna() &
    reviews_clean["Rating"].isna() &
    reviews_clean["Metadata"].isna() &
    reviews_clean["Time"].isna() &
    (reviews_clean["Pictures"] == 0)
)
reviews_clean = reviews_clean.loc[~placeholder].copy()

In [18]:
# Validating removal
print("Placeholder Rows Removed: ", placeholder.sum())
print("Remaining Views: ", reviews_clean.shape[0])

Placeholder Rows Removed:  2
Remaining Views:  9962


##### 3.2.2.1 Handling Missing Review Text

Some records contain missing review text despite having other potentially useful information, such as ratings or picture counts.

Rather than discarding these records entirely, I will preserve them in the cleaned dataset. Missing review text will be excluded from text-based feature extraction when that stage is performed, allowing other available information to remain useful.

#### 3.2.3 Cleaning Rating Column

In [19]:
reviews_clean["Rating"] = pd.to_numeric(reviews_clean["Rating"], errors="coerce")           # Conversion to Numeric

##### 3.2.3.1 Handling Missing Ratings

Missing ratings will not be imputed, as assigning artificial values could distort restaurant-level rating averages.

Records with missing ratings will be retained because they may still contribute to other features. During aggregation, missing ratings will be excluded from rating-based calculations while other available information will remain usable.

#### 3.2.4 Removing Reviewer-Related Columns

The Reviewer and Metadata columns contain information about individual reviewers rather than restaurants. Since these attributes do not directly contribute to restaurant-level segmentation, I will remove them from the working dataset to avoid introducing irrelevant features.

In [20]:
reviews_clean = reviews_clean.drop(columns=["Reviewer", "Metadata"])

#### 3.2.5 Converting Review Timestamps

The Time column contains the dates and times at which reviews were submitted. I will convert this column to a datetime format to enable temporal feature extraction, such as identifying review patterns and aggregating review activity over time.

The original timestamp information will be retained, as it may be useful when deriving restaurant-level temporal features.

In [21]:
reviews_clean["Time"] = pd.to_datetime(reviews_clean["Time"], errors="coerce")

#### 3.2.6 Retaining Pictures

The Pictures column is already numerical and contains no missing values. I will retain it without further transformation, as picture counts may provide a useful indicator of review engagement when aggregated at the restaurant level.

#### 3.2.7 Validation of the Cleaned Dataset

In [22]:
print("Cleaned Dataset Shape: ", reviews_clean.shape)
print("\nRemaining Columns: ", reviews_clean.columns.tolist())
print("\nDuplicate Rows: ", reviews_clean.duplicated().sum())
print("\nMissing Ratings: ", reviews_clean["Rating"].isna().sum())
print("\nTIme Data Type: ", reviews_clean["Time"].dtype)
print("\nPictures Data Type: ", reviews_clean["Pictures"].dtype)

Cleaned Dataset Shape:  (9962, 5)

Remaining Columns:  ['Restaurant', 'Review', 'Rating', 'Time', 'Pictures']

Duplicate Rows:  0

Missing Ratings:  1

TIme Data Type:  datetime64[us]

Pictures Data Type:  int64


### 3.3 Summary

The reviews dataset was cleaned by removing exact duplicate records and placeholder entries, converting ratings and timestamps into appropriate formats, and removing reviewer-related columns that do not contribute to restaurant segmentation.

Missing ratings were retained without imputation, review text was preserved for subsequent text-based feature extraction, and picture counts were left unchanged. These decisions ensure that useful information is preserved while preparing the dataset for restaurant-level aggregation and feature engineering.

## 4. Review Aggregation
### 4.1 Rating-Based Feature

Customer ratings provide an indication of overall satisfaction with a restaurant. I will aggregate the ratings for each restaurant to calculate its average rating and rating variability.

I will also retain the number of valid ratings as a supporting measure. Since the dataset contains a fixed sample of reviews per restaurant, rating counts will not be treated as a measure of restaurant popularity.

In [23]:
rating_features = reviews_clean.groupby("Restaurant").agg(
    avg_rating = ("Rating", "mean"),
    rating_std = ("Rating", "std"),
    rating_count = ("Rating", "count")
).reset_index()
display(rating_features.head())

,Restaurant,avg_rating,rating_std,rating_count
0,10 Downing Street,3.80,1.091751,100
1,13 Dhaba,3.48,1.566570,100
2,"3B's - Buddies, Bar & Barbecue",4.76,0.830237,100
3,AB's - Absolute Barbecues,4.88,0.498077,100
4,Absolute Sizzlers,3.62,1.581330,100


### 4.2 Review Text Features

Review text can provide additional information about customer engagement and the level of detail in their feedback. I will calculate the word count of each review and aggregate it into the median review length for each restaurant.

This feature represents the typical amount of written feedback customers provide rather than the sentiment or quality of their experiences.

In [24]:
# Review Word Count
reviews_clean["review_wc"] = (
    reviews_clean["Review"]
    .str.split()
    .str.len()
)

In [25]:
# Review Length
text_features = reviews_clean.groupby("Restaurant").agg(
    median_rl = ("review_wc", "median")
).reset_index()
display(text_features.head())

,Restaurant,median_rl
0,10 Downing Street,50.5
1,13 Dhaba,32.0
2,"3B's - Buddies, Bar & Barbecue",27.0
3,AB's - Absolute Barbecues,29.5
4,Absolute Sizzlers,34.0


### 4.3 Picture-Based Features

Picture counts may provide an indication of customer engagement and how frequently customers share visual content about their dining experiences.

I will aggregate picture counts for each restaurant to calculate the average number of pictures per review and the total number of pictures shared. These features will help capture differences in customer sharing behavior.

In [26]:
picture_ft = reviews_clean.groupby("Restaurant").agg(
    avg_pic = ("Pictures", "mean"), 
    total_pic = ("Pictures", "sum")
).reset_index()
display(picture_ft.head())

,Restaurant,avg_pic,total_pic
0,10 Downing Street,1.05,105
1,13 Dhaba,0.41,41
2,"3B's - Buddies, Bar & Barbecue",0.13,13
3,AB's - Absolute Barbecues,0.24,24
4,Absolute Sizzlers,0.45,45


### 4.4 Consolidating Review Features

I have extracted rating, review length, and picture-related features separately. I will now combine them into a single restaurant-level dataset.

I will retain only the features that provide meaningful information for restaurant segmentation and exclude supporting counts or redundant variables that are not suitable for clustering.

In [27]:
# Combining/Merging Features
review_ft = rating_features.merge(
    text_features, on= "Restaurant"
).merge(
    picture_ft, on= "Restaurant"
)

# Removing Redundant Features
review_ft = review_ft.drop(
    columns=["rating_count", "total_pic"]
)
display(review_ft.head())

,Restaurant,avg_rating,rating_std,median_rl,avg_pic
0,10 Downing Street,3.80,1.091751,50.5,1.05
1,13 Dhaba,3.48,1.566570,32.0,0.41
2,"3B's - Buddies, Bar & Barbecue",4.76,0.830237,27.0,0.13
3,AB's - Absolute Barbecues,4.88,0.498077,29.5,0.24
4,Absolute Sizzlers,3.62,1.581330,34.0,0.45


## 5. Merging Datasets

I have prepared the restaurant metadata and aggregated review features separately. I will now combine them into a single restaurant-level dataset using restaurant names as the common key.

I will retain all restaurants from the metadata dataset, including those without available review-derived features, so that no restaurant entities are unnecessarily discarded.

### 5.1 Restaurant Metadata and Review Features

In [28]:
restaurant_df = metadata_clean.merge(
    review_ft,
    left_on="Name",
    right_on="Restaurant",
    how="left",
    validate="one_to_one"
)

# Standardize Restaurant Column
restaurant_df = restaurant_df.drop(columns=["Restaurant"]).rename(
    columns={"Name": "Restaurant"}
)

display(restaurant_df.head())


,Restaurant,Cost,Collections,Cuisines,avg_rating,rating_std,median_rl,avg_pic
0,Beyond Flavours,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...",4.28,1.239746,32.0,0.39
1,Paradise,800,Hyderabad's Hottest,"Biryani, North Indian, Chinese",4.70,0.771984,28.0,0.02
2,Flechazo,"1,300","Great Buffets, Hyderabad's Hottest","Asian, Mediterranean, North Indian, Desserts",4.66,0.878704,30.5,0.55
3,Shah Ghouse Hotel & Restaurant,800,Late Night Restaurants,"Biryani, North Indian, Chinese, Seafood, Bever...",3.21,1.603626,32.5,0.30
4,Over The Moon Brew Company,"1,200","Best Bars & Pubs, Food Hygiene Rated Restauran...","Asian, Continental, North Indian, Chinese, Med...",4.34,1.065719,36.0,0.44


### 5.2 Validating the Merged Dataset

I will verify that the merged dataset contains the expected number of restaurants, that restaurant names remain unique, and that any missing review-derived features are identified.

Restaurants without available review data will retain their metadata, while their review-derived features will remain missing until I decide how to handle them during preprocessing.

In [29]:
print("Dataset Shape:", restaurant_df.shape)
print("Duplicate Restaurants:", restaurant_df["Restaurant"].duplicated().sum())
print("Missing Restaurant Names:", restaurant_df["Restaurant"].isna().sum())
print("\nMissing Review Features:")
print(restaurant_df[["avg_rating", "rating_std", "median_rl", "avg_pic"]].isna().sum())

Dataset Shape: (105, 8)
Duplicate Restaurants: 0
Missing Restaurant Names: 0

Missing Review Features:
avg_rating    5
rating_std    5
median_rl     5
avg_pic       5
dtype: int64


## 6. Final Dataset

### 6.1 Final Dataset Validation

The restaurant metadata and review-derived features have been successfully combined into a single dataset containing 105 restaurants.

The merged dataset contains restaurant cost, cuisine, collection information, average customer ratings, rating variability, median review length, and average picture counts.

Five restaurants do not have review-derived features, while the metadata has been retained for all restaurants. These missing values will be evaluated and handled during the preprocessing stage before clustering.

The dataset is now ready for further preprocessing and feature transformation.

### 6.2 Saving the Prepared Dataset

In [30]:
processed_data_path = data_path / "processed"
processed_data_path.mkdir(parents=True, exist_ok=True)
restaurant_df.to_csv(
    processed_data_path / "restaurant_features.csv",
    index=False
)
print("Dataset successfully saved.")

Dataset successfully saved.
